# RAG Assignment — International Scholarship Q&A System
**Name:** Samrawit Yisak

## Topic
This project builds a Retrieval-Augmented Generation (RAG) system that answers
questions about the **eligibility, application procedures and deadlines** of major
international scholarships (Chevening, Fulbright, Gates Cambridge, Rhodes,
Commonwealth, Knight-Hennessy, DAAD, Eiffel, and others).

**Who it serves:** prospective international students who need accurate,
source-cited information before applying.

**Key constraint of this assignment:** no Hugging Face models are used.
Everything runs locally with **ChromaDB's built-in ONNX embedding model**
(all-MiniLM-L6-v2 via ONNX Runtime), and generation is **extractive** — the bot
only quotes real sentences from the collected documents and always cites its
sources. When a question is outside the corpus, the system **abstains** instead
of inventing an answer.



In [1]:
import os, sys, chromadb
print("working dir:", os.getcwd())
print("python:", sys.executable)
print("chromadb:", chromadb.__version__)

working dir: /home/Student/rag_scholarship_bot
python: /usr/bin/python3.9
chromadb: 1.5.9


## Task 1 — Data Collection 

We need ~100 real pages of scholarship info.

- **Part a:** ~40 hand-picked official URLs (eligibility, deadlines, how to apply).
- **Part b:** a small web crawler expands to 100 by following scholarship-related
  links on the same sites.

**Ground rules:** polite (0.4s delay), only real HTML pages, minimum 200 words,
no overwriting. Every file is saved with its **source URL, terms link and crawl
date** — so every answer can be traced back later.

In [13]:
import os, re, time
from datetime import date
from urllib.parse import urljoin, urldefrag
from collections import deque
import requests
from bs4 import BeautifulSoup

RAW_DIR = "data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; ScholarshipRAGBot/1.0; student assignment)"}
TIMEOUT = 15
MIN_WORDS = 200          
TARGET = 100            

CURATED = [
    # Fulbright
    "https://foreign.fulbrightonline.org/",
    "https://us.fulbrightonline.org/countries/united-states",
    # Chevening
    "https://www.chevening.org/scholarships/",
    "https://www.chevening.org/scholarships/chevening-scholarship/",
    "https://www.chevening.org/scholarships/eligibility/",
    "https://www.chevening.org/scholarships/how-to-apply/",
    # Erasmus+
    "https://erasmus-plus.ec.europa.eu/opportunities/opportunities-for-individuals/students/",
    # DAAD
    "https://www.daad.de/en/studying-in-germany/scholarships/",
    "https://www2.daad.de/deutschland/stipendium/datenbank/en/21148-scholarship-database/",
    # Commonwealth
    "https://cscuk.fcdo.gov.uk/scholarships/commonwealth-masters-scholarships/",
    "https://cscuk.fcdo.gov.uk/about-us/scholarships/",
    # Gates Cambridge
    "https://www.gatescambridge.org/apply/",
    "https://www.gatescambridge.org/about/",
    # Rhodes
    "https://www.rhodeshouse.ox.ac.uk/scholarships/the-rhodes-scholarship/",
    "https://www.rhodeshouse.ox.ac.uk/scholarships/apply-for-a-rhodes-scholarship/",
    # Knight-Hennessy
    "https://knighthennessy.stanford.edu/admission",
    "https://knighthennessy.stanford.edu/admission/apply",
    "https://knighthennessy.stanford.edu/about",
    # Clarendon
    "https://www.ox.ac.uk/clarendon/scholarships-and-fees/clarendon-scholarships",
    "https://www.ox.ac.uk/admissions/graduate/fees-and-funding/fees-and-funding-overview",
    # MEXT Japan
    "https://www.studyinjapan.go.jp/en/planning/scholarship/",
    "https://www.studyinjapan.go.jp/en/planning/scholarship/type-j/",
    "https://www.studyinjapan.go.jp/en/planning/scholarship/type-a/",
    # Global Korea Scholarship
    "https://www.studyinkorea.go.kr/en/sub/gks/gks_invitation.do",
    # Eiffel France
    "https://www.campusfrance.org/en/eiffel-scholarship-program-of-excellence",
    "https://www.campusfrance.org/en/scholarships",
    # Holland Scholarship
    "https://www.studyinnl.org/finances/nl-scholarship",
    # Swiss Excellence
    "https://www.sbfi.admin.ch/sbfi/en/home/education/scholarships-and-grants/swiss-government-excellence-scholarships.html",
    # JASSO
    "https://www.jasso.go.jp/en/study_j/scholarship/index.html",
    # Portals / funding info
    "https://scholarshipportal.com/scholarships",
    "https://www.mastersportal.com/scholarships/",
    "https://studyineurope.eu/scholarships",
    # University aid pages
    "https://www.ox.ac.uk/admissions/graduate/fees-and-funding/graduate-scholarships",
    "https://www.cam.ac.uk/study/international-students/fees-and-funding",
    "https://www.imperial.ac.uk/study/fees-and-funding/postgraduate-doctoral/scholarships-and-studentships/",
    "https://www.manchester.ac.uk/study/masters/fees-and-funding/scholarships/",
    "https://www.tum.de/en/studies/fees-and-scholarships",
]


def fetch(url):
    """Download a page and return (text, terms_hint) or (None, None) on failure."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT, allow_redirects=True)
        if r.status_code != 200 or "text/html" not in r.headers.get("Content-Type", ""):
            return None, None
        soup = BeautifulSoup(r.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside", "form", "noscript"]):
            tag.decompose()  # strip non-content elements
        text = re.sub(r"\n{3,}", "\n\n", soup.get_text("\n", strip=True))
        if len(text.split()) < MIN_WORDS:
            return None, None  
    
        a = soup.find("a", string=re.compile("terms|privacy|legal", re.I))
        terms = urljoin(url, a["href"]) if (a and a.get("href")) else "https://" + r.url.split("/")[2] + "/terms"
        return text, terms
    except Exception:
        return None, None

def save(url, text, terms):
    """Write one .txt file with the required header block."""
    name = re.sub(r"[^a-zA-Z0-9]+", "_", url.split("//", 1)[1])[:60].rstrip("_")
    path = os.path.join(RAW_DIR, name + ".txt")
    if os.path.exists(path):
        return False
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"=== SOURCE ===\n{url}\n\n=== LICENSE / TERMS ===\n{terms}\n\n"
                f"=== CRAWLED ===\n{date.today().isoformat()}\n\n=== CONTENT ===\n{text}\n")
    return True


log, seen = [], set()
def crawl(url):
    key = urldefrag(url)[0]
    if key in seen:
        return
    seen.add(key)
    text, terms = fetch(key)
    if text:
        save(key, text, terms)
        log.append(("OK  ", key))
    else:
        log.append(("SKIP", key))
    time.sleep(0.4)  

for u in CURATED:
    crawl(u)

def domain(u):
    parts = urldefrag(u)[0].split("/")
    return parts[2] if len(parts) > 2 else ""

frontier = deque([u for (st, u) in log if st.startswith("OK")])
while len(log) < TARGET and frontier:
    base = frontier.popleft()
    try:
        r = requests.get(base, headers=HEADERS, timeout=TIMEOUT)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception:
        continue
    for a in soup.find_all("a", href=True):
        if len(log) >= TARGET:
            break
        child = urljoin(base, a["href"].split("#")[0])
        # SAFETY FIRST: check it's a real http URL before calling domain()
        if (child.startswith("http")
                and domain(child) == domain(base)
                and child not in seen
                and re.search(r"scholar|fellow|fund|eligib|applic|deadline|admiss|grant|award", child, re.I)):
            crawl(child)

ok = sum(1 for st, _ in log if st.startswith("OK"))
print(f"Tried {len(log)} URLs -> {ok} saved to data/raw/")
for st, u in log:
    print(st, u)


Tried 100 URLs -> 71 saved to data/raw/
SKIP https://foreign.fulbrightonline.org/
SKIP https://us.fulbrightonline.org/countries/united-states
OK   https://www.chevening.org/scholarships/
OK   https://www.chevening.org/scholarships/chevening-scholarship/
OK   https://www.chevening.org/scholarships/eligibility/
OK   https://www.chevening.org/scholarships/how-to-apply/
OK   https://erasmus-plus.ec.europa.eu/opportunities/opportunities-for-individuals/students/
SKIP https://www.daad.de/en/studying-in-germany/scholarships/
SKIP https://www2.daad.de/deutschland/stipendium/datenbank/en/21148-scholarship-database/
OK   https://cscuk.fcdo.gov.uk/scholarships/commonwealth-masters-scholarships/
OK   https://cscuk.fcdo.gov.uk/about-us/scholarships/
OK   https://www.gatescambridge.org/apply/
OK   https://www.gatescambridge.org/about/
OK   https://www.rhodeshouse.ox.ac.uk/scholarships/the-rhodes-scholarship/
SKIP https://www.rhodeshouse.ox.ac.uk/scholarships/apply-for-a-rhodes-scholarship/
OK   http

### Task 1b — Top-up crawl 

Some programmes were skipped on the first run (bot-blocking or thin pages) —
which made questions like "MEXT deadline" come back empty.

This cell retries a second batch of URLs (Fulbright apply pages, MEXT/Japan,
DAAD, Knight-Hennessy, Eiffel). It reuses `crawl()` from Task 1.

**After running this, re-run Task 2 (cleaning) and Task 3 (embedding)** so the
new pages join the corpus.


In [3]:
EXTRA = [
    "https://foreign.fulbrightonline.org/",
    "https://foreign.fulbrightonline.org/index.php",
    "https://us.fulbrightonline.org/",
    "https://us.fulbrightonline.org/about",
    "https://studyinjapan.go.jp/en/planning/scholarship/",
    "https://www.studyinjapan.go.jp/en/planning/scholarship/procedure/",
    "https://www.jasso.go.jp/en/study_j/scholarship/index.html",
    "https://www.studyinkorea.go.kr/en/sub/gks/allnew_invitation.do",
    "https://www2.daad.de/deutschland/stipendium/datenbank/en/21148-scholarship-database/",
    "https://www.daad.de/en/studying-in-germany/scholarships/",
    "https://www.campusfrance.org/en/scholarships",
]
for u in EXTRA:
    crawl(u)

import os
print("raw files now:", sum(1 for f in os.listdir('data/raw') if f.endswith('.txt')))

EXTRA3 = [
    "https://us.fulbrightonline.org/applicants",
    "https://us.fulbrightonline.org/apply",
    "https://us.fulbrightonline.org/applicants/apply-now",
    "https://foreign.fulbrightonline.org/applicants",
    "https://us.fulbrightonline.org/algorithm/apply/apply-overview",
    "https://us.fulbrightonline.org/timeline-closed",
    "https://knighthennessy.stanford.edu/admission/apply",
    "https://www.daad.de/en/study-and-research-in-germany/scholarships/",
]
for u in EXTRA3:
    crawl(u)
print("raw files now:", sum(1 for f in os.listdir('data/raw') if f.endswith('.txt')))


raw files now: 77
raw files now: 77


## Task 2 — Cleaning 

Raw web pages are messy. For every file in `data/raw/`, this cell:

- **Fixes mojibake** — `â€™` → `'`, `Ã©` → `é`, etc.
- **Normalises Unicode** (NFKC) — fancy spaces and symbols become plain text
- **Deletes boilerplate lines** — cookie banners, "Sign in", "Back to top"...
- **Normalises whitespace** — collapse double spaces, keep paragraphs
- **Drops files under 80 words** — those are menus, not content
- **Removes duplicates** — an MD5 fingerprint of the *sorted* word list catches
  files that are the same page saved twice

The provenance header (source URL, terms, crawl date) is kept untouched —
only the content is cleaned.

The table at the end shows words before/after for every file, plus three
before/after samples used in the report.


In [4]:
import os, re, hashlib, unicodedata
import pandas as pd

RAW_DIR, CLEAN_DIR = "data/raw", "data/clean"
os.makedirs(CLEAN_DIR, exist_ok=True)

# 1) mojibake map: common UTF-8-read-as-Latin1 artefacts -> proper characters
ENCODING_FIXES = {
    "â€™": "'", "â€˜": "'", "â€œ": '"', "â€\x9d": '"',
    "â€“": "–", "â€”": "—", "â€¦": "…", "Â ": " ", "Â": "",
    "Ã©": "é", "Ã¨": "è", "Ã¼": "ü", "Ã¶": "ö", "Ã±": "ñ", "Ã§": "ç", "Ã": "A",
}

# 2) boilerplate: whole lines that are pure website chrome, not content
BOILERPLATE = re.compile(
    r"^(cookie|cookies|accept all|manage preferences|privacy policy|terms(?: of use)?|"
    r"sign in|log in|subscribe|newsletter|back to top|share this|follow us|"
    r"menu|home|about us|contact us|search|all rights reserved|copyright).*",
    re.I,
)

def clean_text(text):
    # fix mojibake
    for bad, good in ENCODING_FIXES.items():
        text = text.replace(bad, good)
    # unicode normalisation (NFKC folds fullwidth chars, fancy spaces, etc.)
    text = unicodedata.normalize("NFKC", text)
    # remove boilerplate lines
    lines = [ln for ln in text.split("\n") if ln.strip() and not BOILERPLATE.match(ln.strip())]
    text = "\n".join(lines)
    # normalise whitespace: collapse runs of spaces, keep paragraph breaks
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def fingerprint(text):
    """MD5 of the SORTED word list — near-duplicate detector."""
    words = sorted(w.lower() for w in re.findall(r"[a-z0-9']+", text.lower()))
    return hashlib.md5(" ".join(words).encode()).hexdigest()

stats, seen_fp = [], {}
for fname in sorted(os.listdir(RAW_DIR)):
    if not fname.endswith(".txt"):
        continue  # skip hidden folders like .ipynb_checkpoints
    raw = open(os.path.join(RAW_DIR, fname), encoding="utf-8").read()
    # split off the header block so cleaning only touches content
    head, _, body = raw.partition("=== CONTENT ===")
    meta = dict(re.findall(r"=== (\w[\w /]*?) ===\n(.*?)\n\n", head + "\n\n", re.S))
    words_before = len(body.split())

    body = clean_text(body)
    words_after = len(body.split())

    if words_after < 80:                                   # too short -> drop
        stats.append((fname, words_before, words_after, "DROPPED: too short"))
        continue
    fp = fingerprint(body)
    if fp in seen_fp:                                      # duplicate -> drop
        stats.append((fname, words_before, words_after, f"DROPPED: duplicate of {seen_fp[fp]}"))
        continue
    seen_fp[fp] = fname
    with open(os.path.join(CLEAN_DIR, fname), "w", encoding="utf-8") as f:
        f.write(head + "=== CONTENT ===\n" + body + "\n")  # keep header block intact
    stats.append((fname, words_before, words_after, "kept"))

# ---- aggregate report ----
df = pd.DataFrame(stats, columns=["file", "words_before", "words_after", "status"])
print(df.to_string(index=False))
kept = df[df.status == "kept"]
print(f"\nkept: {len(kept)}   dropped-short: {(df.status.str.contains('too short')).sum()}   "
      f"dropped-dup: {(df.status.str.contains('duplicate')).sum()}   "
      f"total words removed: {(df.words_before - df.words_after).clip(lower=0).sum()}")

# ---- before/after sample for 3 kept docs (for the report) ----
for fname in kept.file.head(3):
    raw_body = open(os.path.join(RAW_DIR, fname), encoding="utf-8").read().partition("=== CONTENT ===")[2]
    clean_body = open(os.path.join(CLEAN_DIR, fname), encoding="utf-8").read().partition("=== CONTENT ===")[2]
    print(f"\n--- {fname} ---")
    print("BEFORE:", repr(raw_body.strip()[:150]))
    print("AFTER :", repr(clean_body.strip()[:150]))

                                                            file  words_before  words_after                                                            status
                     cscuk_fcdo_gov_uk_about_us_scholarships.txt          1524         1519                                                              kept
     cscuk_fcdo_gov_uk_about_us_scholarships_and_fellowships.txt          1524         1519 DROPPED: duplicate of cscuk_fcdo_gov_uk_about_us_scholarships.txt
                     cscuk_fcdo_gov_uk_advice_for_applicants.txt          1611         1606                                                              kept
cscuk_fcdo_gov_uk_commonwealth_scholarships_frequently_asked.txt          2984         2976                                                              kept
cscuk_fcdo_gov_uk_commonwealth_shared_scholarship_university.txt          3113         3105                                                              kept
              cscuk_fcdo_gov_uk_csc_research_impact_

## Task 3 — Chunks, Embeddings & ChromaDB 

Turning documents into something a machine can search by *meaning*:

1. **Chunk** — each document is split into **200-word pieces with 50-word
   overlap** (the window slides 150 words each time). This way a sentence cut
   at the edge of one chunk still appears complete in the next. Tiny tails
   under 30 words are dropped. Every chunk keeps its metadata: doc name,
   chunk id, part number and the original URL (for citations).

2. **Embed** — ChromaDB's built-in **ONNX MiniLM** (no Hugging Face) turns each
   chunk into a **384-number vector** that captures its meaning.
    First run downloads ~80 MB and takes a few minutes.

3. **Store** — all vectors go into a persistent ChromaDB collection on disk
   (`data/chroma`), indexed in **cosine space**. The collection is rebuilt
   fresh on every run, so re-running never creates duplicates.


In [5]:
import os, re, json, time
import chromadb
from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2

CLEAN_DIR = "data/clean"
CHUNK_WORDS, OVERLAP_WORDS, MIN_CHUNK = 200, 50, 30

# ---------- 1) chunking ----------
def chunk_words(words, size=CHUNK_WORDS, overlap=OVERLAP_WORDS):
    """Sliding window over a word list: 200-word chunks with 50-word overlap."""
    step = size - overlap  # 150 -> each chunk starts 150 words after the previous one
    return [words[i:i + size] for i in range(0, len(words), step) if i + 30 <= len(words)]

chunks = []  # each: {id, text, metadata}
for fname in sorted(os.listdir(CLEAN_DIR)):
    if not fname.endswith(".txt"):
        continue
    raw = open(os.path.join(CLEAN_DIR, fname), encoding="utf-8").read()
    head, _, body = raw.partition("=== CONTENT ===")
    meta = dict(re.findall(r"=== (\w[\w /]*?) ===\n(.*?)\n\n", head + "\n\n", re.S))
    source_url = meta.get("SOURCE", "").strip()
    stem = fname[:-4]
    for i, wds in enumerate(chunk_words(body.split())):
        text = " ".join(wds)
        if len(wds) < MIN_CHUNK:
            continue  # drop tiny tail pieces
        chunks.append({
            "id": f"{stem}_c{i:02d}",
            "text": text,
            "metadata": {
                "doc": fname,
                "chunk_id": f"{stem}_c{i:02d}",
                "part": i,
                "source_url": source_url,
            },
        })

print(f"clean docs: {len(os.listdir(CLEAN_DIR)) - 1}   chunks: {len(chunks)}")
print("example chunk:", json.dumps(chunks[0], indent=2)[:400] if chunks else "NONE")

# ---------- 2) embeddings + storage ----------
client = chromadb.PersistentClient(path="data/chroma")   # persisted to disk
try:
    client.delete_collection("scholarships")  # safe re-runs: start fresh every time
except Exception:
    pass
col = client.get_or_create_collection(
    name="scholarships",
    embedding_function=ONNXMiniLM_L6_V2(),
    metadata={"hnsw:space": "cosine"},   # cosine distance
)

BATCH = 500
t0 = time.time()
for i in range(0, len(chunks), BATCH):
    batch = chunks[i:i + BATCH]
    col.add(
        ids=[c["id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
    )
    print(f"  embedded + stored {i + len(batch)}/{len(chunks)} chunks "
          f"({time.time() - t0:.1f}s elapsed)")
print(f"\nfinal collection count: {col.count()}")

clean docs: 73   chunks: 648
example chunk: {
  "id": "cscuk_fcdo_gov_uk_about_us_scholarships_c00",
  "text": "Scholarships and Fellowships - Commonwealth Scholarship Commission in the UK Scholarships and Fellowships Our work combines sustainable development with the UK national interest, by supporting innovators and leaders of the future from across the Commonwealth and attracting outstanding talent to the UK\u2019s universities. Programm
  embedded + stored 500/648 chunks (13.4s elapsed)
  embedded + stored 648/648 chunks (17.1s elapsed)

final collection count: 648


## Task 4 — Retriever 

The search engine of our bot. `retrieve(query, k)`:

1. embeds the question with the **same MiniLM model** used for the chunks,
2. asks ChromaDB for the **k closest vectors**,
3. returns each hit with its doc, URL, and **cosine distance**.

**Cosine distance** = how far apart two meanings are. **0 = identical,
lower = better.** Good matches usually land below ~0.7.

The three test queries at the bottom also show an honest lesson: when a
programme is missing from the corpus (MEXT), the retriever can't find it —
and later the bot will *refuse to answer* rather than guess.

In [6]:
def retrieve(query, k=5):
    """Return the k most similar chunks for a query."""
    res = col.query(query_texts=[query], n_results=k)
    hits = []
    for i in range(k):
        hits.append({
            "chunk_id": res["ids"][0][i],
            "doc": res["metadatas"][0][i]["doc"],
            "source_url": res["metadatas"][0][i]["source_url"],
            "distance": res["distances"][0][i],   # cosine distance (0 = identical)
            "text": res["documents"][0][i],
        })
    return hits
for q in ["Chevening eligibility", "MEXT deadline", "Fulbright application steps"]:
    print("=" * 70)
    print("QUERY:", q)
    for h in retrieve(q, k=3):
        print(f"  [{h['distance']:.3f}] {h['doc']}  ({h['chunk_id']})")
        print(f"          {h['text'][:120]}...")

QUERY: Chevening eligibility
  [0.297] www_chevening_org_scholarships_eligibility.txt  (www_chevening_org_scholarships_eligibility_c00)
          Chevening eligibility criteria | Chevening Skip to content Chevening eligibility criteria You must meet our eligibility ...
  [0.346] www_chevening_org_fellowships_who_can_apply_fellowships_elig.txt  (www_chevening_org_fellowships_who_can_apply_fellowships_elig_c00)
          Eligibility for a Chevening Fellowship | Chevening Skip to content Fellowships eligibility criteria General eligibility ...
  [0.365] www_chevening_org_scholarships_who_can_apply.txt  (www_chevening_org_scholarships_who_can_apply_c00)
          Who can apply? | Chevening Skip to content Who can apply for a Chevening Scholarship? Chevening Scholarships allow excep...
QUERY: MEXT deadline
  [0.714] www_chevening_org_application_timeline.txt  (www_chevening_org_application_timeline_c01)
          British embassies and high commissions produce a shortlist. These candidates a

### Corpus check 

Quick check: which programmes actually made it into the corpus?
If a programme's files are missing here (e.g. no `fulbright` or `mext`),
questions about it **cannot** be answered — the bot will abstain instead.

In [7]:
import os
for f in sorted(os.listdir("data/clean")):
    if any(k in f.lower() for k in ["fulbright", "japan", "mext", "korea", "campusfrance"]):
        print(f)
print("---- total docs:", len([f for f in os.listdir('data/clean') if f.endswith('.txt')]))


foreign_fulbrightonline_org_applicants.txt
us_fulbrightonline_org_about.txt
us_fulbrightonline_org_applicants.txt
www_campusfrance_org_en_eiffel_scholarship_program_of_excell.txt
---- total docs: 73


## Task 5 — Answering (with a conscience) 

No LLM — the bot answers **extractively**: it quotes real sentences from the
retrieved chunks, each with its source.

**Two honesty gates decide whether to answer at all:**
- **Gate 1 — distance:** best match must be closer than 0.70 in cosine space.
- **Gate 2 — coverage:** that chunk must contain at least 60% of the question's
  key words (with light plural handling).

Fail either → the bot says **"I could not find this"** instead of guessing.
This kills two classic RAG failures: questions that are semantically far away
(Gate 1), and questions that are *near but wrong* — like "F-1 visa fee" once
matching a chunk about visa *costs* in a scholarship budget (Gate 2).

If both gates pass, every sentence in the top-k chunks is ranked by word
overlap with the question (plus vector distance), and the best four become the
answer — each one traceable to its document and URL.

In [8]:
import re

def split_sentences(text):
    return re.split(r"(?<=[.!?])\s+", text)

STOP = set("a an the and or of to in for on with is are be as at by from that this it "
           "you your we our what when how which who whose why do does i my me".split())

def key_terms(query):
    return [w for w in re.findall(r"[a-z0-9']+", query.lower()) if w not in STOP]

def answer(query, k=4, dist_threshold=0.70, coverage=0.6):
    hits = retrieve(query, k=k)
    best = min(h["distance"] for h in hits)

    # ---- GATE 1: vector distance ----
    if best > dist_threshold:
        return (f"I could not find this in the collected scholarship documents — "
                f"closest match was at distance {best:.2f} (threshold {dist_threshold}). "
                "This question is likely outside the corpus.")

 
    terms = key_terms(query)
    best_chunk = next(h for h in hits if h["distance"] == best)
    chunk_words = set(re.findall(r"[a-z0-9']+", best_chunk["text"].lower()))
    covered = sum(1 for t in terms if t in chunk_words)
    if terms and covered / len(terms) < coverage:
        return (f"I could not find this in the collected scholarship documents — "
                f"the closest chunk only covers {covered}/{len(terms)} of the question's "
                f"key terms (needs {coverage:.0%}). Best source was "
                f"{best_chunk['doc']} ({best_chunk['source_url']}).")

   
    q_set = set(terms)
    scored = []
    for h in hits:
        for s in split_sentences(h["text"]):
            if len(s.split()) < 6:
                continue
            s_words = set(re.findall(r"[a-z0-9']+", s.lower()))
            overlap = len(q_set & s_words) / max(len(q_set), 1)
            if overlap > 0:
                scored.append((overlap, h["distance"], s, h))
    scored.sort(key=lambda t: (-t[0], t[1]))

    lines, used = [], set()
    for ov, dist, s, h in scored[:4]:
        if s in used:
            continue
        used.add(s)
        lines.append(f"• {s.strip()}\n  ↳ {h['doc']}  ({h['source_url']})")
    if not lines:
        return ("I found related chunks but no sentence matched your question directly. "
                f"Top source: {hits[0]['doc']} ({hits[0]['source_url']})")
    return "\n\n".join(lines)

# ---------- demo: three in-scope + one out-of-scope ----------
for q in ["What are the eligibility requirements for the Chevening Scholarship?",
          "How do I apply for the Fulbright Foreign Student Program?",
          "When is the deadline for MEXT scholarship applications?",
          "How much is the F-1 student visa fee?"]:
    print("=" * 75)
    print("Q:", q)
    print(answer(q))
    print()

Q: What are the eligibility requirements for the Chevening Scholarship?
• Chevening eligibility criteria | Chevening Skip to content Chevening eligibility criteria You must meet our eligibility criteria to apply for a Chevening Scholarship.
  ↳ www_chevening_org_scholarships_eligibility.txt  (https://www.chevening.org/scholarships/eligibility/)

• Learn more about Chevening’s eligibility and work experience requirements to assess your own eligibility, and find out more about what a Chevening Award covers .
  ↳ www_chevening_org_scholarships_who_can_apply.txt  (https://www.chevening.org/scholarships/who-can-apply/)

• Learn more about Chevening’s eligibility and work experience requirements.
  ↳ www_chevening_org_scholarships.txt  (https://www.chevening.org/scholarships/)

• In this section To be eligible for a Chevening Scholarship you must: Be a citizen of a Chevening-eligible country or territory .
  ↳ www_chevening_org_scholarships_eligibility.txt  (https://www.chevening.org/scholar

## Task 6 — Evaluation 

Time to grade the bot with a **golden test set**: 10 questions, each with the
document that *should* answer it.

**The metrics:**
- **Hit rate @5** — did the expected document appear in the top 5 results?
- **MRR** (Mean Reciprocal Rank) — how *high* the right document ranks
  (1.0 = always first, 0.5 = around rank 2).
- **Abstain check** — the 2 out-of-scope questions must be refused, not answered.

**Results: 5/8 correct (62%), MRR 0.48, 2/2 refusals correct.**
The three misses are analysed in the report: two corpus gaps (blocked sites),
one ranking tie between adjacent pages of the same programme.


In [9]:
import pandas as pd

GOLDEN = [
    ("What are the eligibility requirements for the Chevening Scholarship?", "www_chevening_org_scholarships_eligibility"),
    ("Who can apply for a Chevening Scholarship?",                            "www_chevening_org_scholarships_who_can_apply"),
    ("What does the Gates Cambridge Scholarship cover?",                     "www_gatescambridge_org_programme_the_scholarship"),
    ("How do I apply for the Rhodes Scholarship?",                           "www_rhodeshouse_ox_ac_uk_scholarships_apply_for_a_rhodes_scholarship"),
    ("What is the Knight-Hennessy Scholars admission process?",              "knighthennessy_stanford_edu_admission_apply"),
    ("What are the Commonwealth Masters Scholarships eligibility rules?",    "cscuk_fcdo_gov_uk_scholarships_commonwealth_masters_scholars"),
    ("What scholarships does DAAD offer for studying in Germany?",           "www_daad_de_en_studying_in_germany_scholarships"),
    ("How do I apply for the Fulbright Foreign Student Program?",            "us_fulbrightonline_org_about"),
    ("When is the deadline for MEXT scholarship applications?",              None),   # out-of-corpus: expect abstain
    ("How much is the F-1 student visa fee?",                                None),   # out-of-scope: expect abstain
]

def eval_row(query, expected_doc, k=5):
    hits = retrieve(query, k=k)
    if expected_doc is None:
        a = answer(query)
        return {"query": query, "expected": "ABSTAIN", "got": "abstains" if "could not find" in a else "ANSWERED(!)",
                "rank": None, "hit": "could not find" in a}
    rank = next((i + 1 for i, h in enumerate(hits) if expected_doc in h["doc"]), None)
    return {"query": query, "expected": expected_doc[:40], "got": hits[0]["doc"][:40],
            "rank": rank, "hit": rank is not None}

rows = [eval_row(q, d) for q, d in GOLDEN]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

rr = [1 / r["rank"] for r in rows if r["rank"]]
answered = [r for r in rows if r["expected"] != "ABSTAIN"]
print(f"\nhit rate (top-5): {sum(r['hit'] for r in answered)}/{len(answered)} = "
      f"{sum(r['hit'] for r in answered)/len(answered):.0%}")
print(f"MRR: {sum(rr)/len(answered):.2f}")
print(f"abstain correctness: {sum(r['hit'] for r in rows if r['expected']=='ABSTAIN')}/2")

                                                               query                                 expected                                      got  rank   hit
What are the eligibility requirements for the Chevening Scholarship? www_chevening_org_scholarships_eligibili www_chevening_org_scholarships_eligibili   1.0  True
                          Who can apply for a Chevening Scholarship? www_chevening_org_scholarships_who_can_a www_chevening_org_scholarships_who_can_a   1.0  True
                    What does the Gates Cambridge Scholarship cover? www_gatescambridge_org_programme_the_sch www_gatescambridge_org_current_scholars.   2.0  True
                          How do I apply for the Rhodes Scholarship? www_rhodeshouse_ox_ac_uk_scholarships_ap www_rhodeshouse_ox_ac_uk_scholarships_se   NaN False
             What is the Knight-Hennessy Scholars admission process? knighthennessy_stanford_edu_admission_ap knighthennessy_stanford_edu_admission.tx   NaN False
   What are the Common

In [10]:
def ask(q):
    print("Q:", q)
    print("-" * 70)
    print(answer(q))
ask("What are the eligibility requirements for the Chevening Scholarship?")
ask("What documents do I need for a UK study visa?")

Q: What are the eligibility requirements for the Chevening Scholarship?
----------------------------------------------------------------------
• Chevening eligibility criteria | Chevening Skip to content Chevening eligibility criteria You must meet our eligibility criteria to apply for a Chevening Scholarship.
  ↳ www_chevening_org_scholarships_eligibility.txt  (https://www.chevening.org/scholarships/eligibility/)

• Learn more about Chevening’s eligibility and work experience requirements to assess your own eligibility, and find out more about what a Chevening Award covers .
  ↳ www_chevening_org_scholarships_who_can_apply.txt  (https://www.chevening.org/scholarships/who-can-apply/)

• Learn more about Chevening’s eligibility and work experience requirements.
  ↳ www_chevening_org_scholarships.txt  (https://www.chevening.org/scholarships/)

• In this section To be eligible for a Chevening Scholarship you must: Be a citizen of a Chevening-eligible country or territory .
  ↳ www_cheveni

In [11]:
pip install gradio

Defaulting to user installation because normal site-packages is not writeable
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached semantic_version-2.10.0-py2.py3-none-any.whl.metadata (9.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 575.5 kB/s  0:00:35a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 1.5 MB/s  0:00:031.4 MB/s eta 0:00:01:010m
Using cached semantic_version-2.10.0-py2.py3-none-any.whl (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 1.3 MB/s  0:00:07a 0:00:010:00:01
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)
  Attempting uninstall: websockets
    Found existing installation: websockets 15.0.1
    Uninstalling websockets-15.0.1:
      Successfully uninstalled websockets-15.0.1
  Attempting uninstall: pillow━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━  4/11 [ruff]
    Found existing installation: pillow 11.3.038;114m╸━━━━━━━━━━━━━━━━━━━━━━━━━  4/11 [ruff]
    Uninstalling pillow-11.3.

In [12]:
import gradio as gr

def chat_fn(message, history):
    return answer(message)

demo = gr.ChatInterface(
    chat_fn,
    title="International Scholarship Q&A Bot",
    description=(
        "Ask about eligibility, application procedures and deadlines of major "
        "international scholarships (Chevening, Fulbright, Gates Cambridge, Rhodes, "
        "Commonwealth, DAAD, Eiffel...). Answers are extracted from the collected "
        "official pages, with sources. Out-of-scope questions are refused."
    ),
    examples=[
        "What are the eligibility requirements for the Chevening Scholarship?",
        "What does the Gates Cambridge Scholarship cover?",
        "How do I apply for the Fulbright Foreign Student Program?",
        "How much is a UK visa fee?",          
        "What is the weather in London?",      
    ],
)

demo.launch()  

Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.
